Reinserting Trackers - PV Paiva (old blocks → 64p / 32p)
=
Replaces the old tracker blocks of the PV Paiva electromechanical drawing with the new `EMF_tracker_64p` / `EMF_tracker_32p` blocks. In the old drawing each tracker is drawn as two halves (one at rotation 0, one at rotation 180, same X, 42.5 apart in Y). For each rotation-0 half of `1x64p` and `1x96p_meio`, the script inserts the new block(s) at the tracker centre:
- `1x64p` → one `EMF_tracker_64p`.
- `1x96p_meio` → one `EMF_tracker_64p` plus one `EMF_tracker_32p` above it.

`1x96p_ponta` and `1x32p` blocks are **not** re-inserted by this notebook. The `0B` and `1str+2str` notebooks continue this block-replacement work on later versions of the drawing.

**Workflow overview:**
1. Convert the DWG to DXF and extract all model space entities.
2. Look at the entity counts per block name.
3. Keep only the old tracker blocks.
4. Compute the new insertion points and write the reinsert script.

**DWG prerequisites** (the code assumes all of these, and it doesn't check them):
- **File:** `PV Paiva - Eletromecânico.dwg` in `C:\Users\Usuario\Desktop\projetos\paiva`. The DXF is written to `...\paiva\output`. The ODA File Converter 27.1.0 must be installed at the path set in `DWG_to_DuckDB_ETL.py`.
- **Model space only:** trackers are block references (`INSERT`) placed directly in model space.
- **Block names** (exact, case-sensitive `==`): `1x96p_ponta`, `1x96p_meio`, `1x64p`, `1x32p`. Any other name is ignored.
- **Attributes:** some block in the drawing (the inverters, in Paiva) must have the `QBT`, `DSJ`, `PVM` and `GRP` attributes, because those columns are dropped by name. Without them the `drop` raises `KeyError`.
- **Rotation:** only the halves with rotation ≠ 180 are used, so each tracker must have exactly one half at rotation 0 and one at 180. Trackers at 90 / 270 would also be picked up.
- **Geometry:** the Y offsets (21.251 for `1x64p`, 21.264 and 43.503 for `1x96p_meio`) are specific to these block definitions.
- **Layers:** the new blocks go on the layer of the old block (`EMF_em_pst_XX`), set with `._layer set` before each group of lines.
- **Running the `.scr` in AutoCAD:** each line is a plain command-line `._insert <block> X,Y 1 1 0` (spaces act as Enter), so:
  - the blocks `EMF_tracker_64p` and `EMF_tracker_32p` must already be defined in the drawing;
  - they must **not** have *Scale uniformly* on, because the line answers X scale, Y scale and rotation;
  - if they have attributes, set `ATTREQ = 0`, because the script gives no attribute values;
  - everything is inserted with rotation 0.
  - delete the old `1x64p` / `1x96p_meio` blocks first, otherwise the trackers end up duplicated.

Loading the DWG entities
=
- Copies `PV Paiva - Eletromecânico.dwg` from `dwg_folder` into `temp_folder`, so the ODA File Converter only processes that single file.
- `convert_dwg_to_dxf` (from `DWG_to_DuckDB_ETL.py`) converts it to DXF (ACAD2018) in ``output_folder` (`paiva\output`)`.
- `extract_data_from_dxf` reads every model space entity with `ezdxf` into a list of dicts (no DuckDB here). For block references it keeps the block name, insertion point, rotation, color and every attribute tag as its own key.
- Deletes the DXF and the temp copy of the DWG afterwards.
- `pd` comes from the `*` import of `DWG_to_DuckDB_ETL`. `math` is not used.

In [13]:
from DWG_to_DuckDB_ETL import *
import os
import shutil
import numpy as np
import math

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\paiva"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\temp"
output_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\output"

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Paiva - Eletromecânico.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Paiva - Eletromecânico.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Paiva - Eletromecânico.dwg")

data = extract_data_from_dxf(dxf_file)
print("Data Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Data Loaded


Entity overview
=
Converts the entities into a DataFrame and prints `describe()`, `info()` and the count per block name. In the saved run: 5922 `1x96p_ponta`, 2959 `1x96p_meio`, 2334 `1x64p`, 1152 `1x32p`, 464 inverters and 29 PSTs.

In [14]:
ELM_df = pd.DataFrame(data)

print(ELM_df.describe())
print(ELM_df.info())
print(ELM_df['name'].value_counts())

                  X              Y      rotation
count  12860.000000   12860.000000  12860.000000
mean   35060.961230  128079.732587     25.250389
std      593.173520     951.762144     62.562695
min    33667.553332  125996.367175      0.000000
25%    34622.695128  127293.973600      0.000000
50%    35000.922088  128002.283775      0.000000
75%    35483.685992  128959.993661      0.000000
max    36625.891051  129787.710488    270.000000
<class 'pandas.DataFrame'>
RangeIndex: 12889 entries, 0 to 12888
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   type      12889 non-null  str    
 1   layer     12889 non-null  str    
 2   handle    12889 non-null  str    
 3   name      12860 non-null  str    
 4   X         12860 non-null  float64
 5   Y         12860 non-null  float64
 6   rotation  12860 non-null  float64
 7   UFV       12860 non-null  str    
 8   PST       12860 non-null  str    
 9   INV       12831 non-null  s

Selecting the old tracker blocks
=
- Keeps the four old tracker blocks (`1x96p_ponta`, `1x96p_meio`, `1x64p`, `1x32p`).
- Drops the columns that are not needed (`handle`, `type` and the inverter-only attributes `QBT`, `DSJ`, `PVM`, `GRP`).

In [15]:
mask = (ELM_df['name'] == '1x96p_ponta') | (ELM_df['name'] == '1x96p_meio') | (ELM_df['name'] == '1x64p') | (ELM_df['name'] == '1x32p')
tracker_df = ELM_df[mask].copy()

tracker_df.drop(columns=['handle', 'type', 'QBT', 'DSJ', 'PVM', 'GRP'], inplace=True)

print(tracker_df)
print(tracker_df.describe())
print(tracker_df.info())

               layer   name             X              Y  rotation UFV PST INV
58     EMF_em_pst_21  1x64p  34979.686150  127465.060575     180.0   0  21   9
59     EMF_em_pst_21  1x64p  34979.686150  127507.562575       0.0   0  21   9
60     EMF_em_pst_21  1x64p  34985.986150  127465.060575     180.0   0  21   9
61     EMF_em_pst_21  1x64p  34985.986150  127507.562575       0.0   0  21   9
62     EMF_em_pst_21  1x64p  34992.286150  127465.060575     180.0   0  21   9
...              ...    ...           ...            ...       ...  ..  ..  ..
12884  EMF_em_pst_01  1x32p  35683.005250  129540.154750     180.0   0  01  12
12885  EMF_em_pst_01  1x32p  35689.305250  129540.154750     180.0   0  01  12
12886  EMF_em_pst_01  1x32p  35701.905250  129540.154750     180.0   0  01  12
12887  EMF_em_pst_01  1x32p  35708.205250  129540.154750     180.0   0  01  12
12888  EMF_em_pst_09  1x32p  34127.453648  128845.259495       0.0   0  09  04

[12367 rows x 8 columns]
                  X       

New insertion points and reinsert script
=
- Keeps only `1x64p` and `1x96p_meio` blocks with rotation ≠ 180 (the upper half of each tracker).
  - The mask is built on `ELM_df` but applied to `tracker_df`, which gives the "Boolean Series key will be reindexed" warning. It works because both share the same index.
- New Y:
  - `1x64p` → `Y − 21.251` (the tracker centre, half of the 42.5 gap between the two halves).
  - `1x96p_meio` → `Y − 21.264` for the new `EMF_tracker_64p`, and `Y2 = new Y + 43.503 + 21.264` (= old Y + 43.503) for the extra `EMF_tracker_32p`.
- `Script`: one `._insert EMF_tracker_64p X,Y 1 1 0` line per block, plus the `EMF_tracker_32p` line for `1x96p_meio`.
- Writes the lines grouped by layer, each group after `(command-s "._layer" "set" "<layer>" "")`, to `Desktop\Pessoal\Python\Ecotechne\Reinserting Trackers\Reinserting Trackers 0B.scr`.
  - Warning: this is the same file name the `Reinserting Trackers 0B` notebook writes, so running one overwrites the other's script.

In [16]:
reinsert_mask = (ELM_df['name'] != '1x96p_ponta') & (ELM_df['name'] != '1x32p') & (ELM_df['rotation'] != 180)
reinsert_df = tracker_df[reinsert_mask].copy()

y_fix_64p = 21.251
y_fix_96p_to_64p = 21.264
y_fix_96p_to_32p = 43.503 + 21.264

reinsert_df['Y'] = np.where(
    reinsert_df['name'] == '1x64p',
    reinsert_df['Y']-y_fix_64p,
    reinsert_df['Y']-y_fix_96p_to_64p
)

reinsert_df['Y2'] = np.where(
    reinsert_df['name'] == '1x96p_meio',
    reinsert_df['Y']+y_fix_96p_to_32p,
    0
)

reinsert_df['new_name'] = 'EMF_tracker_64p'

reinsert_df['Script'] = np.where(
    reinsert_df['name'] == '1x96p_meio',
    reinsert_df.apply(
        lambda row:
        f"._insert {row['new_name']} {row['X']},{row['Y']} 1 1 0\n"
        f"._insert EMF_tracker_32p {row['X']},{row['Y2']} 1 1 0",
        axis=1
    ),
    reinsert_df.apply(
        lambda row:
        f"._insert {row['new_name']} {row['X']},{row['Y']} 1 1 0",
        axis=1
    )
)

with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Reinserting Trackers\Reinserting Trackers 0B.scr", 'w') as f:
    for i in reinsert_df['layer'].unique():
        mask = reinsert_df['layer'] == i

        if mask.any():
            f.write(f'(command-s "._layer" "set" "{i}" "")\n')
            f.write('\n'.join(reinsert_df[mask]['Script'].astype(str)) + '\n')



C:\Users\Usuario\AppData\Local\Temp\ipykernel_15996\2252478718.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reinsert_df = tracker_df[reinsert_mask].copy()
